# Grant Webscraper

## Project Structure

In [ ]:
                         +-------------------+
                         |   Config / Setup  |
                         |-------------------|
                         | HEADERS (UA)      |
                         | LOG_PATH          |
                         | keyword_weights   |
                         +-------------------+
                                   |
                                   v
                         +-------------------+
                         |   URL Sources     |
                         |-------------------|
                         | Static listings   |
                         | Google search     |
                         +-------------------+
                                   |
                                   v
                         +-------------------+
                         |   Web Scraper     |
                         |-------------------|
                         | requests + BS4    |
                         | classify page:    |
                         |  - listing page   |
                         |  - detail page    |
                         +-------------------+
                                   |
                  -----------------+-----------------
                  |                                 |
                  v                                 v
     +------------------------+          +------------------------+
     |   Listing Page         |          |   Detail Page          |
     |------------------------|          |------------------------|
     | Find links & paginate  |          | Extract HTML + Text     |
     | Recurse crawl (depth)  |          | Compute scores:         |
     |                        |          |  - keyword_score        |
     |                        |          |  - cosine_similarity    |
     +------------------------+          | Save corpus files:      |
                                         |  - html_path            |
                                         |  - text_path            |
                                         | Build meta row:         |
                                         |  url, doc_id,           |
                                         |  source_domain, etc.    |
                                         | Add "relevance" (blank) |
                                         +------------------------+
                                                      |
                                                      v
                                         +------------------------+
                                         |   Results Collector    |
                                         |------------------------|
                                         | Append rows to list    |
                                         +------------------------+
                                                      |
                                                      v
                                         +------------------------+
                                         |   Meta File Builder    |
                                         |------------------------|
                                         | scored_results.csv     |
                                         | Merge old+new rows     |
                                         | Dedupe by doc_id/url   |
                                         +------------------------+
                                                      |
                                                      v
                                         +------------------------+
                                         |   Corpus on Disk       |
                                         |------------------------|
                                         | logs/corpus/html/*.html|
                                         | logs/corpus/text/*.txt |
                                         | scored_results.csv     |
                                         +------------------------+


## Config & Setup

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import re
import os
import pandas as pd

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/114.0.0.0 Safari/537.36"
}

LOG_PATH = r"C:\\Users\\miked\\Desktop2\\IConnectFoundation\\grantMinded\\logs"
os.makedirs(LOG_PATH, exist_ok=True)

## Keyword and Scoring
will need seperate py file

In [4]:
# Domain-expert keyword weights
keyword_weights = {
    "aging": 3,
    "dementia": 3,
    "isolation": 3,
    "Alzheimer's": 3,
    "telehealth": 2,
    "501(c)(3)": 3,
    "grant": 4,
    "funding": 3
}

# Web search queries
search_keywords = [
    "aging nonprofit funding",
    "Alzheimer's foundation grant",
    "dementia telehealth grant",
    "community isolation grant 501(c)(3)"
]

# Stage 1: Keyword match scoring
def keyword_score(text, keywords):
    score = 0
    matched = []
    text_lower = text.lower()
    for word, weight in keywords.items():
        pattern = rf'\b{re.escape(word.lower())}\b'
        if re.search(pattern, text_lower):
            score += weight
            matched.append(word)
    return score, matched

# Stage 2: Cosine similarity with stemming via TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import nltk

nltk.download('punkt')

stemmer = PorterStemmer()

def stemmed_tokenizer(text):
    tokens = word_tokenize(text.lower())
    return [stemmer.stem(token) for token in tokens if token.isalpha()]

reference_text = """
health care contracts and partnerships that improve health outcomes and quality of life for older adults, 
people with disabilities and/or caregivers. Award recipients are recognized for their bold, 
transformative, innovative initiatives designed to align health and social care and increase their organization’s sustainability.
"""

def cosine_score(grant_text, ref_text=reference_text):
    docs = [ref_text, grant_text]
    vectorizer = TfidfVectorizer(stop_words='english', tokenizer=stemmed_tokenizer)
    tfidf_matrix = vectorizer.fit_transform(docs)
    similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
    return similarity


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\miked\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## Scraper Tools

### Webscapers

In [6]:
# specific website list review
def scrape_website(url, timeout=15):
    try:
        response = requests.get(url, headers=HEADERS, timeout=timeout)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        text = soup.get_text(separator=' ', strip=True)
        return url, text, soup
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return url, "", None

# keyword list review
def google_search(query, num_results=15):
    from googlesearch import search
    urls = []
    try:
        for url in search(query, num_results=num_results):
            urls.append(url)
    except Exception as e:
        print(f"Google search error for '{query}': {e}")
    return urls
    
def is_grant_detail_page(url, soup):
    """
    Detect if the page is a grant detail page.
    Looks for keywords like 'deadline', 'due date', 'eligibility', 'award', 'description' 
    in the visible text, and/or URL patterns.
    """
    if soup is None:
        return False

    text = soup.get_text(separator=' ', strip=True).lower()

    # Keywords often on grant detail pages
    detail_keywords = ['deadline', 'due date', 'eligibility', 'award', 'description', 'grant details', 'application', 'funding amount']

    # If any detail keyword found in page text -> likely grant detail page
    if any(kw in text for kw in detail_keywords):
        return True

    # Also check URL patterns that suggest a detail page (customize as needed)
    detail_url_patterns = ['grant-details', 'opportunity', 'funding-opportunity', 'award']

    if any(pattern in url.lower() for pattern in detail_url_patterns):
        return True

    return False

def is_listing_page(url, soup):
    """
    Detect if page is a listing page, showing multiple grants/opportunities.
    For example, pages with lots of links with keywords like 'grant', 'funding', 'opportunity' in URLs.
    """
    if soup is None:
        return False

    links = soup.find_all('a', href=True)
    count_relevant_links = 0
    for a in links:
        href = a['href'].lower()
        if any(keyword in href for keyword in ['grant', 'funding', 'opportunity']):
            count_relevant_links += 1

    # Heuristic: If many relevant links (>5), treat as listing page
    return count_relevant_links >= 5



### Recrusive Grant Scrape

In [8]:
def recursive_scrape_grant_links(start_url, base_url=None, depth=2, min_score=1, visited=None):
    """
    Recursively scrape starting from start_url, visiting pages up to `depth` levels deep.
    Save grant detail pages with keyword + cosine scores for dataset building.
    Includes pagination handling: crawls all relevant links before moving to next pages.
    """
    if visited is None:
        visited = set()
    results = []

    def crawl(url, current_depth):
        if current_depth == 0 or url in visited:
            return
        visited.add(url)
        print(f"Visiting {url} (depth {current_depth})")

        try:
            url, text, soup = scrape_website(url)
            if not soup:
                return

            # Check if grant detail page
            if is_grant_detail_page(url, soup):
                k_score, matched = keyword_score(text, keyword_weights)
                c_score = cosine_score(text)
                print(f"  Detail page found with keyword_score={k_score}, cosine_score={round(c_score, 3)}, matched={matched}")

                results.append({
                    'url': url,
                    'content_snippet': text[:2000],
                    'keyword_score': k_score,
                    'matched_keywords': ', '.join(matched),
                    'cosine_similarity': round(c_score, 4),
                    'valid_label': 'unlabeled',
                    'label_notes': '',
                    'granting_organization': '',
                    'date_of_loi_due': '',
                    'date_of_submission': ''
                })
                return

            # If it's a listing page, crawl all relevant links before pagination
            elif is_listing_page(url, soup):
                next_pages = []
                grant_links = []

                for a in soup.find_all('a', href=True):
                    href = a['href']
                    link = urljoin(url, href)
                    text = a.get_text(strip=True).lower()

                    if base_url and not link.startswith(base_url):
                        continue

                    # Pagination detection (next/page 2/>> or ?page=2/start=)
                    if re.search(r'(next|page\s*\d+|>>)', text) or re.search(r'page=\d+|start=\d+', link):
                        next_pages.append(link)
                    else:
                        grant_links.append(link)

                # Crawl relevant-looking links first
                for link in grant_links:
                    crawl(link, current_depth - 1)

                # Then crawl pagination links
                for link in next_pages:
                    crawl(link, current_depth - 1)

        except Exception as e:
            print(f"Error crawling {url}: {e}")

    crawl(start_url, depth)
    return results


## Pipeline Run

In [10]:
def run_pipeline(min_score=1, depth=3):
    results = []

    static_urls = [
        "https://new-york.thegrantportal.com/aging-seniors",
        "https://www.grants.gov/learn-grants/grant-eligibility.html",
        "https://www.eda.gov/funding/funding-opportunities/all-opportunities?f%5B0%5D=funding_status%3A6565",
        "https://www.walmart.org/how-we-give/program-guidelines/spark-good-local-grants-guidelines",
        "https://submit.gatesfoundation.org/",
        "https://www.rwjf.org/en/grants/active-funding-opportunities.html"
    ]

    # Scrape static URLs
    for url in static_urls:
        print(f"\nScraping listing site: {url}")
        grant_records = recursive_scrape_grant_links(url, base_url=url, depth=depth, min_score=min_score)
        results.extend(grant_records)

    # Google search scraping
    for query in search_keywords:
        print(f"\nSearching Google for: '{query}'")
        found_urls = google_search(query, num_results=50)
        for url in found_urls:
            print(f"Visiting {url} for grant opportunities...")
            grant_records = recursive_scrape_grant_links(url, base_url=url, depth=depth, min_score=min_score)
            results.extend(grant_records)

    # Save or update CSV log with all fields
    df_new = pd.DataFrame(results)
    csv_path = os.path.join(LOG_PATH, 'scored_results.csv')

    if os.path.exists(csv_path): # might want to update to include loi dates or other unique features of new funding announcements
        df_old = pd.read_csv(csv_path)
        df_combined = pd.concat([df_old, df_new], ignore_index=True)
        df_combined = df_combined.drop_duplicates(subset=['url'])
    else:
        df_combined = df_new

    os.makedirs(LOG_PATH, exist_ok=True)
    df_combined.to_csv(csv_path, index=False)

    print(f"\n✅ Saved updated scored results to {csv_path}")


In [12]:
run_pipeline(min_score=1)


Scraping listing site: https://new-york.thegrantportal.com/aging-seniors
Visiting https://new-york.thegrantportal.com/aging-seniors (depth 3)


C:\Users\miked\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
C:\Users\miked\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:408: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['abov', 'afterward', 'alon', 'alreadi', 'alway', 'ani', 'anoth', 'anyon', 'anyth', 'anywher', 'becam', 'becaus', 'becom', 'befor', 'besid', 'cri', 'describ', 'dure', 'els', 'elsewher', 'empti', 'everi', 'everyon', 'everyth', 'everywher', 'fifti', 'formerli', 'forti', 'ha', 'henc', 'hereaft', 'herebi', 'hi', 'howev', 'hundr', 'inde', 'latterli', 'mani', 'meanwhil', 'moreov', 'mostli', 'nobodi', 'noon', 'noth', 'nowher', 'onc', 'onli', 'otherwis', 'ourselv', 'perhap', 'pleas', 'seriou', 'sever', 'sinc', 'sincer', 'sixti', 'someon', 'someth', 'sometim', 'somewher', 'themselv', 'thenc', 'thereaft', 'there

  Detail page found with keyword_score=13, cosine_score=0.07, matched=['aging', "Alzheimer's", 'grant', 'funding']

Scraping listing site: https://www.grants.gov/learn-grants/grant-eligibility.html
Visiting https://www.grants.gov/learn-grants/grant-eligibility.html (depth 3)
  Detail page found with keyword_score=7, cosine_score=0.037, matched=['grant', 'funding']

Scraping listing site: https://www.eda.gov/funding/funding-opportunities/all-opportunities?f%5B0%5D=funding_status%3A6565
Visiting https://www.eda.gov/funding/funding-opportunities/all-opportunities?f%5B0%5D=funding_status%3A6565 (depth 3)
  Detail page found with keyword_score=7, cosine_score=0.018, matched=['grant', 'funding']

Scraping listing site: https://www.walmart.org/how-we-give/program-guidelines/spark-good-local-grants-guidelines
Visiting https://www.walmart.org/how-we-give/program-guidelines/spark-good-local-grants-guidelines (depth 3)
  Detail page found with keyword_score=7, cosine_score=0.114, matched=['grant'

PermissionError: [Errno 13] Permission denied: 'C:\\\\Users\\\\miked\\\\Desktop2\\\\IConnectFoundation\\\\grantMinded\\\\logs\\scored_results.csv'